# CAISO Load Forecasting — Model Analysis

**Best model: GAM** (Semi-parametric Additive Model, Fan & Hyndman 2010)

Average 2025 MAPE by model:
| Model | Avg MAPE | Notes |
|---|---|---|
| **GAM** | **~7.9%** | Best — wins every month except July |
| XGBoost | ~19% | Second-best, fast |
| Hinge (SVR) | ~23% | Decent baseline |
| Transformer | ~37% | Very inconsistent with default params |
| LSTM | ~55% | Needs much more tuning |
| Linear/Ridge | >100% | Fails — temperature relationship is nonlinear |

**Why GAM wins:** California load has a sharp nonlinear temperature response (AC load). GAM fits cubic splines to temperature, capturing the hockey-stick curve that linear models miss entirely.

**Paper reference:** Fan & Hyndman (2010), *Short-term load forecasting based on a semi-parametric additive model*, IEEE Trans. Power Systems. Claims 1.88% MAPE on Australian NEM.

**Gap to paper:** ~6pp (7.9% vs 1.88%). Main reason: paper fits **separate models per half-hour period** (48 models), we use one global model.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import warnings
warnings.filterwarnings('ignore')

DATA_PATH = '../data/processed/filtered.csv'
METRICS_PATH = '../outputs/model_runs/comparison_outputs/monthly_metrics_summary.csv'

df = pd.read_csv(DATA_PATH)
df['time_utc'] = pd.to_datetime(df['time_utc'])
print(f'Dataset: {len(df):,} rows | {df["region"].nunique()} regions | '
      f'{df["time_utc"].min().date()} to {df["time_utc"].max().date()}')
print('Regions:', df['region'].unique().tolist())

## 1. Existing 2025 results — all models compared

In [ ]:
metrics = pd.read_csv(METRICS_PATH)

pivot = metrics.pivot_table(index='model', columns='predict_month', values='MAPE').round(1)
pivot['mean'] = pivot.mean(axis=1).round(1)
pivot = pivot.sort_values('mean')

print('MAPE (%) by model and month — lower is better')
display(pivot.style
    .background_gradient(cmap='RdYlGn_r', axis=None, vmin=0, vmax=50)
    .format('{:.1f}')
)

In [ ]:
# Line plot of monthly MAPE
plot_models = ['gam', 'xgboost', 'hinge_regression', 'transformer', 'lstm']
colors = {'gam': '#2196F3', 'xgboost': '#FF9800', 'hinge_regression': '#9C27B0',
          'transformer': '#F44336', 'lstm': '#4CAF50'}

fig, ax = plt.subplots(figsize=(14, 5))
for m in plot_models:
    row = metrics[metrics['model'] == m].sort_values('month_ts')
    ax.plot(row['predict_month'], row['MAPE'], marker='o', label=m,
            color=colors.get(m), linewidth=2)

ax.set_ylabel('MAPE (%)')
ax.set_xlabel('Month')
ax.set_title('Monthly MAPE — 2025 validation')
ax.legend()
ax.tick_params(axis='x', rotation=45)
ax.axhline(y=5, color='gray', linestyle='--', alpha=0.5, label='5% target')
ax.set_ylim(0, 80)
plt.tight_layout()
plt.show()

## 2. Train and examine the GAM model

Requires: `pip install pygam`

The GAM model from Fan & Hyndman (2010) fits:
```
log(load) = spline(temperature) + cyclic_spline(hour) + cyclic_spline(day_of_year)
          + linear(humidity, cloud, wind, precip, is_weekend, holidays)
          + spline(load_previous_week) + spline(load_lag_24h)
          + factor(region) + factor(day_of_week) + noise
```

In [ ]:
from src.modeling.train_forecaster import (
    set_seed, prepare_base_data, build_feature_matrices,
    prepare_gam_data, build_gam_terms, GAM_FEATURE_COLUMNS,
    compute_metrics, pretty_print_metrics,
)
import argparse

try:
    from pygam import LinearGAM
    print('pygam available')
except ImportError:
    print('pygam not installed — run: pip install pygam')

PREDICT_MONTH = '2025-06'   # change to any YYYY-MM

set_seed(42)
train_df, valid_df, month_start, next_month_start, train_end = prepare_base_data(df, PREDICT_MONTH)
print(f'Train: {len(train_df):,} rows | Validation: {len(valid_df):,} rows')

In [ ]:
# Prepare GAM-specific features (adds load_lag_24h, load_24h_avg, region_code)
train_s = train_df.sort_values(['region', 'time_utc']).reset_index(drop=True)
valid_s = valid_df.sort_values(['region', 'time_utc']).reset_index(drop=True)

X_tr, X_va, y_tr, y_va, feat_cols, gam_train_df, gam_valid_df = prepare_gam_data(train_s, valid_s)
print(f'Features ({len(feat_cols)}): {feat_cols}')
print(f'Train rows after lag drop: {len(X_tr):,} | Valid rows: {len(X_va):,}')

In [ ]:
# Fit GAM (~30–60 seconds on CPU)
terms = build_gam_terms(feat_cols, n_splines=25, lam=0.6)
gam = LinearGAM(terms, max_iter=100, verbose=False)
gam.fit(X_tr, y_tr)

valid_preds = np.expm1(gam.predict(X_va))
valid_preds = np.maximum(valid_preds, 0.0)

metrics_gam = compute_metrics(y_va, valid_preds)
pretty_print_metrics(metrics_gam)

## 3. Prediction vs Actual

In [ ]:
REGIONS = gam_valid_df['region'].unique()

fig, axes = plt.subplots(len(REGIONS), 1, figsize=(16, 3.5 * len(REGIONS)), sharex=False)
if len(REGIONS) == 1:
    axes = [axes]

ev = gam_valid_df[['region', 'time_utc', 'load_mw']].copy()
ev['predicted'] = valid_preds
ev['time_utc'] = pd.to_datetime(ev['time_utc'])

for ax, region in zip(axes, REGIONS):
    sub = ev[ev['region'] == region].copy()
    mape = np.mean(np.abs((sub['load_mw'] - sub['predicted']) / sub['load_mw'].replace(0, np.nan))) * 100
    ax.plot(sub['time_utc'], sub['load_mw'], label='Actual', linewidth=1)
    ax.plot(sub['time_utc'], sub['predicted'], label='GAM Predicted', linewidth=1, alpha=0.85)
    ax.set_title(f'{region} — MAPE: {mape:.1f}%')
    ax.set_ylabel('Load (MW)')
    ax.legend(loc='upper right', fontsize=8)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d'))
    ax.xaxis.set_major_locator(mdates.WeekdayLocator(interval=1))

plt.suptitle(f'GAM: Actual vs Predicted — {PREDICT_MONTH}', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## 4. GAM interpretability — partial effects (what the model learned)

This is the key advantage of GAM over tree/neural models: we can see exactly what each feature contributes.

In [ ]:
# Plot partial effects for interpretable features
PLOT_TERMS = {
    'temperature_2m': 'Temperature (°F)',
    'hour': 'Hour of Day',
    'day_of_year': 'Day of Year',
    'load_previous_week': 'Previous Week Load (MW)',
    'load_lag_24h': '24h Lag Load (MW)',
}

fig, axes = plt.subplots(1, len(PLOT_TERMS), figsize=(4 * len(PLOT_TERMS), 4))

for ax, (feat, xlabel) in zip(axes, PLOT_TERMS.items()):
    if feat not in feat_cols:
        ax.set_title(f'{feat} not in model')
        continue
    idx = feat_cols.index(feat)
    XX = gam.generate_X_grid(term=idx)
    pdep, confi = gam.partial_dependence(term=idx, X=XX, width=0.95)

    ax.plot(XX[:, idx], pdep, color='#2196F3', linewidth=2)
    ax.fill_between(XX[:, idx], confi[:, 0], confi[:, 1], alpha=0.2, color='#2196F3')
    ax.set_xlabel(xlabel, fontsize=9)
    ax.set_ylabel('Partial effect on log(load)')
    ax.set_title(feat, fontsize=9)
    ax.axhline(0, color='gray', linewidth=0.8, linestyle='--')

plt.suptitle('GAM Partial Effects — What each feature contributes', fontsize=12)
plt.tight_layout()
plt.show()
print('The temperature curve should show the U-shape: higher load at cold AND hot extremes (heating + AC).')
print('Hour of day should peak mid-afternoon (CAISO peak ~4-7pm).')

## 5. Compare GAM vs XGBoost on same month

XGBoost is the second-best model and much faster to run.

In [ ]:
from src.modeling.train_forecaster import build_feature_matrices, fit_and_predict
import argparse

prepared = build_feature_matrices(train_s, valid_s)

def quick_args(model):
    return argparse.Namespace(
        model=model, predict_month=PREDICT_MONTH, seed=42,
        ridge_alpha=10.0,
        svr_c=1.0, svr_epsilon=0.1, svr_max_iter=5000,
        xgb_n_estimators=300, xgb_learning_rate=0.05, xgb_max_depth=6,
        xgb_subsample=0.8, xgb_colsample_bytree=0.8,
        xgb_reg_alpha=0.0, xgb_reg_lambda=1.0,
        xgb_min_child_weight=1.0, xgb_tree_method='hist',
        rf_n_estimators=300, rf_max_depth=0, rf_min_samples_leaf=1,
        lgbm_n_estimators=300, lgbm_learning_rate=0.05, lgbm_max_depth=-1,
        lgbm_num_leaves=31, lgbm_subsample=0.8, lgbm_colsample_bytree=0.8,
        lgbm_reg_alpha=0.0, lgbm_reg_lambda=1.0,
        gam_n_splines=25, gam_lam=0.6, gam_max_iter=100,
        lookback=24, epochs=10, batch_size=128, lr=1e-3, weight_decay=1e-5,
        hidden_dim=64, num_layers=2, dropout=0.1,
        d_model=64, nhead=4, dim_feedforward=128, cnn_channels=64, device='auto',
    )

xgb_results = fit_and_predict(quick_args('xgboost'), prepared)
print('XGBoost MAPE:', f"{xgb_results['metrics']['MAPE']:.2f}%")
print('GAM MAPE:    ', f"{metrics_gam['MAPE']:.2f}%")

In [ ]:
# Side-by-side comparison for one region
REGION = gam_valid_df['region'].iloc[0]

gam_sub = ev[ev['region'] == REGION].copy()

xgb_ev = xgb_results['eval_valid_df'].copy()
xgb_ev['predicted'] = xgb_results['valid_preds']
xgb_ev['time_utc'] = pd.to_datetime(xgb_ev['time_utc'])
xgb_sub = xgb_ev[xgb_ev['region'] == REGION].copy()

fig, axes = plt.subplots(2, 1, figsize=(16, 7), sharex=True)

for ax, (sub, name, color) in zip(axes, [
    (gam_sub, 'GAM (Fan & Hyndman)', '#2196F3'),
    (xgb_sub, 'XGBoost', '#FF9800'),
]):
    mape = np.mean(np.abs((sub['load_mw'] - sub['predicted']) / sub['load_mw'].replace(0, np.nan))) * 100
    ax.plot(sub['time_utc'], sub['load_mw'], label='Actual', color='black', linewidth=1.2)
    ax.plot(sub['time_utc'], sub['predicted'], label=name, color=color, linewidth=1.2, alpha=0.85)
    ax.fill_between(sub['time_utc'],
                    sub['load_mw'], sub['predicted'],
                    alpha=0.15, color=color)
    ax.set_title(f'{name} — {REGION} — MAPE: {mape:.1f}%', fontsize=11)
    ax.set_ylabel('Load (MW)')
    ax.legend()
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d'))
    ax.xaxis.set_major_locator(mdates.WeekdayLocator(interval=1))

plt.suptitle(f'GAM vs XGBoost — {REGION} — {PREDICT_MONTH}', fontsize=13)
plt.tight_layout()
plt.show()

## 6. Key performance analysis — when does GAM struggle?

July 2025 is the GAM outlier (18.98% vs ~7% all other months). Investigate why.

In [ ]:
# Load GAM July predictions if they exist
import os
july_pred_path = '../outputs/model_runs/gam_2025-07_predictions.csv'
if os.path.exists(july_pred_path):
    july = pd.read_csv(july_pred_path)
    july['time_utc'] = pd.to_datetime(july['time_utc'])
    
    fig, axes = plt.subplots(2, 1, figsize=(16, 7))
    
    # Prediction vs actual
    ax = axes[0]
    r = july['region'].iloc[0]
    sub = july[july['region'] == r]
    ax.plot(sub['time_utc'], sub['actual_load_mw'], label='Actual', linewidth=1)
    ax.plot(sub['time_utc'], sub['predicted_load_mw'], label='GAM', linewidth=1, alpha=0.85)
    ax.set_title(f'July 2025 — {r} (MAPE ~19%, worst month)')
    ax.set_ylabel('Load (MW)')
    ax.legend()
    
    # APE by hour — does it fail at certain hours?
    ax2 = axes[1]
    july['hour'] = july['time_utc'].dt.hour
    july['ape'] = np.abs((july['actual_load_mw'] - july['predicted_load_mw']) / july['actual_load_mw'].replace(0, np.nan)) * 100
    july.groupby('hour')['ape'].mean().plot(ax=ax2, marker='o', color='#F44336')
    ax2.set_title('Mean APE by hour of day (July 2025) — peak AC hours should show largest error')
    ax2.set_xlabel('Hour')
    ax2.set_ylabel('Mean APE (%)')
    ax2.axhline(y=5, color='gray', linestyle='--', alpha=0.5)
    
    plt.tight_layout()
    plt.show()
else:
    print(f'File not found: {july_pred_path}')
    print('Run: bash scripts/train_forecasters.sh to generate results for all months')

## 7. Per-hour GAM — the key improvement from Fan & Hyndman

The paper fits **24 separate models** (one per hour of day). Each learns its own temperature response. This is the most important enhancement not yet in our implementation.

This cell runs per-hour GAMs on a single predict_month and shows the improvement.

In [ ]:
# Per-hour GAM implementation (Fan & Hyndman approach)
# Fits one model per hour of day — mirrors the paper's 48-period approach

from pygam import LinearGAM, s, l, f
from src.modeling.train_forecaster import build_features, drop_bad_rows

def fit_per_hour_gam(train_df, valid_df, n_splines=20, lam=0.6, max_iter=50):
    """Fan & Hyndman (2010): separate model per hour of day."""
    HOURS = sorted(train_df['hour'].unique())
    
    FEAT = ['temperature_2m', 'apparent_temperature', 'load_previous_week',
            'relative_humidity_2m', 'shortwave_radiation',
            'is_weekend', 'US_federal_holidays', 'state_holidays',
            'doy_sin', 'doy_cos']
    CAT = ['region']
    
    region_map = {r: i for i, r in enumerate(sorted(train_df['region'].unique()))}
    train_df = train_df.copy()
    valid_df = valid_df.copy()
    train_df['region_code'] = train_df['region'].map(region_map).fillna(0).astype(int)
    valid_df['region_code'] = valid_df['region'].map(region_map).fillna(0).astype(int)
    
    all_cols = FEAT + ['region_code']
    
    results = []
    for h in HOURS:
        tr_h = train_df[train_df['hour'] == h].dropna(subset=['load_mw'] + FEAT)
        va_h = valid_df[valid_df['hour'] == h].dropna(subset=FEAT)
        if tr_h.empty or va_h.empty:
            continue
        
        X_tr = tr_h[all_cols].fillna(tr_h[all_cols].median())
        y_tr = np.log1p(tr_h['load_mw'].values)
        X_va = va_h[all_cols].fillna(X_tr.median())
        
        # Build terms: splines for continuous, factor for region
        terms = None
        for i, col in enumerate(all_cols):
            t = f(i) if col == 'region_code' else s(i, n_splines=n_splines, lam=lam)
            terms = t if terms is None else terms + t
        
        model = LinearGAM(terms, max_iter=max_iter, verbose=False)
        model.fit(X_tr.values, y_tr)
        preds = np.expm1(model.predict(X_va.values))
        
        for i, (idx, row) in enumerate(va_h.iterrows()):
            results.append({
                'time_utc': row['time_utc'],
                'region': row['region'],
                'actual_load_mw': row.get('load_mw', np.nan),
                'predicted_load_mw': max(0, preds[i]),
                'hour': h,
            })
    
    return pd.DataFrame(results)

print(f'Fitting per-hour GAM models for {PREDICT_MONTH}... (~1-2 minutes)')
train_feat = build_features(train_s)
valid_feat = build_features(valid_s)
train_feat = drop_bad_rows(train_feat, 'TRAIN_PH')
valid_feat = drop_bad_rows(valid_feat, 'VALID_PH')

per_hour_results = fit_per_hour_gam(train_feat, valid_feat)

ph_valid = per_hour_results.dropna(subset=['actual_load_mw'])
ph_mape = np.mean(np.abs((ph_valid['actual_load_mw'] - ph_valid['predicted_load_mw']) /
                          ph_valid['actual_load_mw'].replace(0, np.nan))) * 100

print(f'\nPer-hour GAM MAPE: {ph_mape:.2f}%')
print(f'Global GAM MAPE:   {metrics_gam["MAPE"]:.2f}%')
print(f'Improvement:       {metrics_gam["MAPE"] - ph_mape:.2f} percentage points')

In [ ]:
# Per-hour GAM vs Global GAM vs Actual
REGION = valid_df['region'].iloc[0]

ph_sub = per_hour_results[per_hour_results['region'] == REGION].copy()
ph_sub['time_utc'] = pd.to_datetime(ph_sub['time_utc'])
ph_sub = ph_sub.sort_values('time_utc')

fig, ax = plt.subplots(figsize=(16, 5))
ax.plot(gam_sub['time_utc'], gam_sub['load_mw'], label='Actual', color='black', linewidth=1.5)
ax.plot(gam_sub['time_utc'], gam_sub['predicted'], label='Global GAM', color='#2196F3', linewidth=1, alpha=0.8)
ax.plot(ph_sub['time_utc'], ph_sub['predicted_load_mw'], label='Per-Hour GAM (Fan & Hyndman)', color='#E91E63', linewidth=1, alpha=0.8)
ax.set_title(f'Global GAM vs Per-Hour GAM — {REGION} — {PREDICT_MONTH}')
ax.set_ylabel('Load (MW)')
ax.legend()
ax.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d'))
ax.xaxis.set_major_locator(mdates.WeekdayLocator(interval=1))
plt.tight_layout()
plt.show()

## 8. Performance target summary

| Benchmark | MAPE | Notes |
|---|---|---|
| Fan & Hyndman paper | **1.88%** | Australian NEM, half-hourly, single region, per-period models |
| Our per-hour GAM | see above | Closer to paper approach |
| Our global GAM | **~7.9%** | Current best |
| XGBoost | **~19%** | Second-best, fast |
| Realistic target | **3–5%** | With per-hour models + multi-site temps |

**Main remaining gaps vs the paper:**
1. Paper uses 2 temperature sites per region + temperature differential features → better captures spatial variation
2. Paper uses lags from last 2–6 *same-period days* specifically, not just lag-24h
3. Paper uses temperature max/min/avg in last 24h — we partially have this via `apparent_temperature`
4. We have multiple regions → need region-specific temperature data